# Part A

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## Load Data

In [2]:
df = pd.read_csv('AI_SocialMedia_Student_Health_Dataset_clean.csv')
print(df.shape)
df.head()

(15000, 14)


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Social_Isolation_Score,Burnout_Level,Academic_Performance_Score,Academic_Failure_Risk
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90,4.05,Low,88.80,0
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94,4.65,Moderate,50.60,0
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75,5.28,Low,96.67,0
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38,3.66,Low,85.65,0
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04,3.25,Moderate,71.84,0


In [3]:
df.dtypes

Student_ID                        str
Age                             int64
Gender                            str
Education_Level                   str
Daily_Social_Media_Hours      float64
Daily_AI_Tool_Usage_Hours     float64
Sleep_Hours                   float64
Physical_Activity_Hours       float64
Mental_Health_Score           float64
Physical_Health_Score         float64
Social_Isolation_Score        float64
Burnout_Level                     str
Academic_Performance_Score    float64
Academic_Failure_Risk           int64
dtype: object

## Descriptive Statistics

In [4]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,15000.0,19.04,3.77,13.00,16.00,19.00,22.00,25.00
Daily_Social_Media_Hours,15000.0,4.54,2.40,0.00,2.83,4.50,6.18,14.00
Daily_AI_Tool_Usage_Hours,15000.0,2.58,1.68,0.00,1.31,2.51,3.74,9.50
Sleep_Hours,15000.0,6.54,1.27,2.00,5.68,6.55,7.41,11.15
Physical_Activity_Hours,15000.0,1.25,1.04,0.00,0.31,1.13,1.95,5.00
Mental_Health_Score,15000.0,72.51,9.25,32.56,66.78,73.76,79.58,91.76
Physical_Health_Score,15000.0,88.02,9.68,48.03,81.52,89.47,97.16,99.98
Social_Isolation_Score,15000.0,4.31,1.16,0.97,3.47,4.28,5.12,8.41
Academic_Performance_Score,15000.0,78.31,11.54,23.99,70.53,78.59,86.70,99.98
Academic_Failure_Risk,15000.0,0.06,0.24,0.00,0.00,0.00,0.00,1.00


## Missing Values and Duplicates

In [5]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Student_ID:', df['Student_ID'].duplicated().sum())

Missing values per column:
Student_ID                    0
Age                           0
Gender                        0
Education_Level               0
Daily_Social_Media_Hours      0
Daily_AI_Tool_Usage_Hours     0
Sleep_Hours                   0
Physical_Activity_Hours       0
Mental_Health_Score           0
Physical_Health_Score         0
Social_Isolation_Score        0
Burnout_Level                 0
Academic_Performance_Score    0
Academic_Failure_Risk         0
dtype: int64

Duplicate rows: 0
Duplicate Student_ID: 0


## Categorical Values

In [6]:
for col in ['Gender', 'Education_Level', 'Burnout_Level']:
    print(df[col].value_counts())
    print()

Gender
Male          7265
Female        7143
Non-binary     592
Name: count, dtype: int64

Education_Level
High School    5693
University     4710
College        4597
Name: count, dtype: int64

Burnout_Level
Low         9000
Moderate    4200
High        1350
Severe       450
Name: count, dtype: int64



## Validity Checks

In [7]:
hour_cols = ['Daily_Social_Media_Hours', 'Daily_AI_Tool_Usage_Hours', 'Sleep_Hours', 'Physical_Activity_Hours']
score_cols = ['Mental_Health_Score', 'Physical_Health_Score', 'Academic_Performance_Score']

print('Hour values outside [0, 24]:', ((df[hour_cols] < 0) | (df[hour_cols] > 24)).sum().sum())
print('Score values outside [0, 100]:', ((df[score_cols] < 0) | (df[score_cols] > 100)).sum().sum())
print('Rows with total daily hours > 24:', (df[hour_cols].sum(axis=1) > 24).sum())
print('Negative values in numeric columns:', (df.select_dtypes('number') < 0).sum().sum())

Hour values outside [0, 24]: 0
Score values outside [0, 100]: 0
Rows with total daily hours > 24: 0
Negative values in numeric columns: 0


In [8]:
df.groupby('Education_Level')['Age'].agg(['min', 'max', 'count'])

,min,max,count
Education_Level,,,
College,18,21,4597
High School,13,17,5693
University,22,25,4710


## Outliers

In [9]:
num_cols = ['Age'] + hour_cols + ['Mental_Health_Score', 'Physical_Health_Score', 'Social_Isolation_Score', 'Academic_Performance_Score']

q1 = df[num_cols].quantile(0.25)
q3 = df[num_cols].quantile(0.75)
iqr = q3 - q1
n_out = ((df[num_cols] < q1 - 1.5 * iqr) | (df[num_cols] > q3 + 1.5 * iqr)).sum()
pd.DataFrame({'n_outliers': n_out,
              'pct': (n_out / len(df) * 100).round(2),
              'min': df[num_cols].min(),
              'max': df[num_cols].max()})

,n_outliers,pct,min,max
Age,0,0.00,13.00,25.00
Daily_Social_Media_Hours,45,0.30,0.00,14.00
Daily_AI_Tool_Usage_Hours,54,0.36,0.00,9.50
Sleep_Hours,96,0.64,2.00,11.15
Physical_Activity_Hours,51,0.34,0.00,5.00
Mental_Health_Score,175,1.17,32.56,91.76
Physical_Health_Score,59,0.39,48.03,99.98
Social_Isolation_Score,33,0.22,0.97,8.41
Academic_Performance_Score,56,0.37,23.99,99.98


## Zero and Ceiling Values

In [10]:
print('Values equal to 0:')
print((df[hour_cols] == 0).sum())
print()
print('Values >= 99:')
print((df[score_cols] >= 99).sum())

Values equal to 0:
Daily_Social_Media_Hours      533
Daily_AI_Tool_Usage_Hours    1240
Sleep_Hours                     0
Physical_Activity_Hours      2639
dtype: int64

Values >= 99:
Mental_Health_Score              0
Physical_Health_Score         1240
Academic_Performance_Score     299
dtype: int64


## Target Distribution

In [11]:
print(df['Burnout_Level'].value_counts(normalize=True).round(3))
print()
print(df['Academic_Failure_Risk'].value_counts(normalize=True).round(3))
print()
print(df['Academic_Performance_Score'].describe().round(2))

Burnout_Level
Low         0.60
Moderate    0.28
High        0.09
Severe      0.03
Name: proportion, dtype: float64

Academic_Failure_Risk
0    0.94
1    0.06
Name: proportion, dtype: float64

count    15000.00
mean        78.31
std         11.54
min         23.99
25%         70.53
50%         78.59
75%         86.70
max         99.98
Name: Academic_Performance_Score, dtype: float64


## Leakage Check

In [12]:
pd.crosstab(df['Burnout_Level'], df['Academic_Failure_Risk'])

Academic_Failure_Risk,0,1
Burnout_Level,,
High,900,450
Low,9000,0
Moderate,4200,0
Severe,0,450


In [13]:
df.groupby('Academic_Failure_Risk')['Academic_Performance_Score'].agg(['min', 'mean', 'max']).round(2)

,min,mean,max
Academic_Failure_Risk,,,
0,37.36,79.18,99.98
1,23.99,64.79,99.48


## Encoding

In [14]:
data = df.drop(columns=['Student_ID'])
burnout_map = {'Low': 0, 'Moderate': 1, 'High': 2, 'Severe': 3}
data['Burnout_Level'] = data['Burnout_Level'].map(burnout_map)
data = pd.get_dummies(data, columns=['Gender', 'Education_Level'], drop_first=True, dtype=int)
print(data.shape)
data.head()

(15000, 15)


,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Social_Isolation_Score,Burnout_Level,Academic_Performance_Score,Academic_Failure_Risk,Gender_Male,Gender_Non-binary,Education_Level_High School,Education_Level_University
0,19,5.32,1.21,7.30,1.41,74.06,98.90,4.05,0,88.80,0,1,0,0,0
1,16,5.21,3.89,7.81,2.48,76.13,89.94,4.65,1,50.60,0,0,1,1,0
2,25,3.61,2.65,6.34,0.06,65.01,76.75,5.28,0,96.67,0,1,0,0,1
3,23,2.93,1.43,6.12,1.37,74.67,87.38,3.66,0,85.65,0,1,0,0,1
4,20,6.52,1.64,5.23,1.14,67.74,88.04,3.25,1,71.84,0,0,1,0,0


## Train/Test Split

In [15]:
strata = data['Burnout_Level'].astype(str) + '_' + data['Academic_Failure_Risk'].astype(str)
train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=42, stratify=strata)
data['split'] = 'train'
data.loc[test_idx, 'split'] = 'test'
print(data['split'].value_counts())

split
train    12000
test      3000
Name: count, dtype: int64


In [16]:
pd.concat([
    data.groupby('split')['Burnout_Level'].value_counts(normalize=True).unstack().round(3),
    data.groupby('split')['Academic_Failure_Risk'].value_counts(normalize=True).unstack().round(3).add_prefix('Risk_')
], axis=1)

,0,1,2,3,Risk_0,Risk_1
split,,,,,,
test,0.6,0.28,0.09,0.03,0.94,0.06
train,0.6,0.28,0.09,0.03,0.94,0.06


## Standardization

In [17]:
feature_cols = ['Age'] + hour_cols + ['Mental_Health_Score', 'Physical_Health_Score', 'Social_Isolation_Score']
is_train = data['split'] == 'train'

scaler = StandardScaler()
scaler.fit(data.loc[is_train, feature_cols])

data_scaled = data.copy()
data_scaled[feature_cols] = scaler.transform(data[feature_cols])
data_scaled.loc[is_train, feature_cols].describe().loc[['mean', 'std']].round(2)

,Age,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score,Social_Isolation_Score
mean,-0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Save Outputs

In [18]:
data.to_csv('processed_data.csv', index=False)
data_scaled.to_csv('processed_data_scaled.csv', index=False)
print(data.shape, data_scaled.shape)

(15000, 16) (15000, 16)
